In [1]:
pip install torch torchvision transformers pillow pandas tqdm jiwer evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.3 MB/s eta 0:00:00


In [2]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
from transformers import (
    VisionEncoderDecoderModel,
    TrOCRProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
BASE_FOLDER = "/content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset"
TRAIN_FOLDER = BASE_FOLDER + "/Training"
VAL_FOLDER = BASE_FOLDER + "/Validation"
TEST_FOLDER = BASE_FOLDER + "/Testing"

In [5]:
df_train = pd.read_csv(TRAIN_FOLDER + "/training_labels.csv")
df_train.drop(columns=["GENERIC_NAME"], inplace=True)
df_train

,IMAGE,MEDICINE_NAME
0,0.png,Aceta
1,1.png,Aceta
2,2.png,Aceta
3,3.png,Aceta
4,4.png,Aceta
...,...,...
3115,3115.png,Zithrin
3116,3116.png,Zithrin
3117,3117.png,Zithrin
3118,3118.png,Zithrin


In [6]:
df_val = pd.read_csv(VAL_FOLDER + "/validation_labels.csv")
df_val.drop(columns=["GENERIC_NAME"], inplace=True)
df_val

,IMAGE,MEDICINE_NAME
0,0.png,Aceta
1,1.png,Aceta
2,2.png,Aceta
3,3.png,Aceta
4,4.png,Aceta
...,...,...
775,775.png,Zithrin
776,776.png,Zithrin
777,777.png,Zithrin
778,778.png,Zithrin


In [7]:
df_test = pd.read_csv(TEST_FOLDER + "/testing_labels.csv")
df_test.drop(columns=["GENERIC_NAME"], inplace=True)
df_test

,IMAGE,MEDICINE_NAME
0,0.png,Aceta
1,1.png,Aceta
2,2.png,Aceta
3,3.png,Aceta
4,4.png,Aceta
...,...,...
775,775.png,Zithrin
776,776.png,Zithrin
777,777.png,Zithrin
778,778.png,Zithrin


In [8]:
!pip -q install -U transformers accelerate
!pip -q install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 22.9 MB/s eta 0:00:00


In [9]:
import os, glob, re
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

import evaluate
from jiwer import wer

MODEL_NAME = "microsoft/trocr-base-handwritten"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
MAX_NEW_TOKENS = 96

In [10]:
def normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)  # remove punctuation
    return s



processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

cer_metric = evaluate.load("cer")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
from PIL import Image
import os

TRAIN_IMG_DIR = os.path.join(TRAIN_FOLDER, "training_words")
VAL_IMG_DIR   = os.path.join(VAL_FOLDER,   "validation_words")
TEST_IMG_DIR  = os.path.join(TEST_FOLDER,  "testing_words")

SPLIT_TO_DIR = {
    "train": TRAIN_IMG_DIR,
    "val":   VAL_IMG_DIR,
    "test":  TEST_IMG_DIR,
}

def load_rgb(img_name: str, split_name: str):
    img_path = os.path.join(SPLIT_TO_DIR[split_name], img_name)
    return Image.open(img_path).convert("RGB")



def evaluate_split(df: pd.DataFrame, split_name: str):
    if len(df) == 0:
        return None, None

    preds_raw = []
    gts_raw = df["MEDICINE_NAME"].astype(str).tolist()

    for start in tqdm(range(0, len(df), BATCH_SIZE), desc=f"TrOCR {split_name}"):
        batch = df.iloc[start:start+BATCH_SIZE]

        images = [load_rgb(p, split_name) for p in batch["IMAGE"].tolist()]

        inputs = processor(images=images, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

        batch_preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
        preds_raw.extend(batch_preds)

    out = df.copy()
    out["split"] = split_name
    out["pred_raw"] = preds_raw
    out["pred_norm"] = out["pred_raw"].apply(normalize_text)
    out["gt_norm"] = out["MEDICINE_NAME"].apply(normalize_text)

    exact_match = (out["pred_norm"] == out["gt_norm"]).mean()
    cer_value = cer_metric.compute(predictions=out["pred_norm"].tolist(),
                                   references=out["gt_norm"].tolist())
    wer_value = wer(out["gt_norm"].tolist(), out["pred_norm"].tolist())

    metrics = {
        "split": split_name,
        "samples": len(out),
        "exact_match": float(exact_match),
        "cer": float(cer_value),
        "wer": float(wer_value),
    }
    return metrics, out

all_metrics = []
all_outputs = []

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    metrics, out = evaluate_split(df, split_name)
    if metrics is None:
        print(f"[WARN] No samples found for split: {split_name}")
        continue
    all_metrics.append(metrics)
    all_outputs.append(out)

metrics_df = pd.DataFrame(all_metrics)
print("\n===== SUMMARY =====")
print(metrics_df.to_string(index=False))

for out in all_outputs:
    split = out["split"].iloc[0]
    out_csv = os.path.join(BASE_FOLDER, f"trocr_predictions_{split}.csv")
    out.to_csv(out_csv, index=False)
    print("Saved:", out_csv)


combined = pd.concat(all_outputs, ignore_index=True)
combined_csv = os.path.join(BASE_FOLDER, "trocr_predictions_all_splits.csv")
combined.to_csv(combined_csv, index=False)
print("Saved:", combined_csv)


metrics_csv = os.path.join(BASE_FOLDER, "trocr_metrics_summary.csv")
metrics_df.to_csv(metrics_csv, index=False)
print("Saved:", metrics_csv)


TrOCR test: 100%|██████████| 98/98 [06:32<00:00,  4.01s/it]



===== SUMMARY =====
split  samples  exact_match      cer      wer
train     3120     0.088462 0.460010 0.939557
  val      780     0.110256 0.430691 0.924051
 test      780     0.133333 0.430488 0.924051
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/trocr_predictions_train.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/trocr_predictions_val.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/trocr_predictions_test.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/trocr_predictions_all_splits.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD data